# Medium + A2Ultra Blend Submission v3

Memory-conscious final submission notebook.

It trains two LightGBM models sequentially:

- `medium`, fixed at 537 trees
- `a2ultra`, fixed at 868 trees

Predictions are blended 50/50. Local validation blend improved stability from about `0.7017` to `0.7055`.

In [ ]:
from __future__ import annotations

import gc
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl
from pandas.api.types import is_float_dtype, is_integer_dtype

warnings.filterwarnings("ignore")

KAGGLE_ROOT_CANDIDATES = [
    Path("/kaggle/input/home-credit-credit-risk-model-stability"),
    Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability"),
]


def find_local_data_dir() -> Path:
    candidates = [Path("data"), Path.cwd() / "data"]
    candidates.extend(parent / "data" for parent in Path.cwd().resolve().parents)
    for candidate in candidates:
        if (candidate / "sample_submission.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find competition data.")


def find_kaggle_data_dir() -> Path | None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for sample_path in input_root.rglob("sample_submission.csv"):
        data_dir = sample_path.parent
        if (data_dir / "parquet_files" / "train" / "train_base.parquet").exists():
            return data_dir
    return None


KAGGLE_ROOT = next((path for path in KAGGLE_ROOT_CANDIDATES if path.exists()), None)
ROOT = KAGGLE_ROOT if KAGGLE_ROOT is not None else (find_kaggle_data_dir() or find_local_data_dir())
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/kaggle_blend_v3")
WORKING.mkdir(parents=True, exist_ok=True)

sample_submission = pd.read_csv(ROOT / "sample_submission.csv")
DRY_RUN = sample_submission.shape[0] == 10
SAMPLE_ROWS = 50000 if DRY_RUN else 0

MAX_RSS_GB = 30.0
MIN_AVAILABLE_GB = 8.0

print({"root": str(ROOT), "working": str(WORKING), "dry_run": DRY_RUN, "sample_rows": SAMPLE_ROWS})

In [ ]:
import os
import sys


class MemoryLimitExceeded(RuntimeError):
    pass


def _process_memory_mb() -> float | None:
    try:
        import psutil

        return psutil.Process(os.getpid()).memory_info().rss / 1024**2
    except Exception:
        pass

    if sys.platform.startswith("linux") or sys.platform == "darwin":
        try:
            import resource

            rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            return rss / 1024 if sys.platform.startswith("linux") else rss / 1024**2
        except Exception:
            return None
    return None


def _available_memory_gb() -> float | None:
    try:
        import psutil

        return psutil.virtual_memory().available / 1024**3
    except Exception:
        return None


def log_memory(label: str) -> None:
    rss = _process_memory_mb()
    available = _available_memory_gb()
    parts = [f"[memory] {label}"]
    if rss is not None:
        parts.append(f"rss={rss:.1f} MB")
    if available is not None:
        parts.append(f"available={available:.1f} GB")
    print(" | ".join(parts))


def check_memory(
    label: str,
    max_rss_gb: float = 30.0,
    min_available_gb: float = 8.0,
) -> None:
    rss_mb = _process_memory_mb()
    available_gb = _available_memory_gb()
    log_memory(label)

    if rss_mb is not None and rss_mb / 1024 > max_rss_gb:
        raise MemoryLimitExceeded(
            f"Memory guard stopped at {label}: process RSS {rss_mb / 1024:.1f} GB "
            f"> max_rss_gb={max_rss_gb:.1f} GB"
        )
    if available_gb is not None and available_gb < min_available_gb:
        raise MemoryLimitExceeded(
            f"Memory guard stopped at {label}: available memory {available_gb:.1f} GB "
            f"< min_available_gb={min_available_gb:.1f} GB"
        )

log_memory('initialized')

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import polars as pl


SPECIAL_COLUMNS = {"case_id", "num_group1", "num_group2"}


@dataclass(frozen=True)
class FeaturePreset:
    depth0_groups: tuple[str, ...]
    aggregate_groups: tuple[str, ...]
    lite_large_groups: tuple[str, ...] = ()


PRESETS: dict[str, FeaturePreset] = {
    "static": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(),
    ),
    "baseline": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
    ),
    "medium": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_a_1",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
    ),
    "a2lite": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_a_1",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
        lite_large_groups=("credit_bureau_a_2",),
    ),
    "a2core": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "applprev_1",
            "tax_registry_a_1",
            "credit_bureau_a_1",
        ),
        lite_large_groups=("credit_bureau_a_2",),
    ),
    "a2ultra": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "applprev_1",
            "tax_registry_a_1",
        ),
        lite_large_groups=("credit_bureau_a_1", "credit_bureau_a_2"),
    ),
    "full": FeaturePreset(
        depth0_groups=("static_0", "static_cb_0"),
        aggregate_groups=(
            "person_1",
            "person_2",
            "applprev_1",
            "applprev_2",
            "debitcard_1",
            "deposit_1",
            "other_1",
            "tax_registry_a_1",
            "tax_registry_b_1",
            "tax_registry_c_1",
            "credit_bureau_a_1",
            "credit_bureau_a_2",
            "credit_bureau_b_1",
            "credit_bureau_b_2",
        ),
    ),
}


LITE_LARGE_NUMERIC_COLUMNS: dict[str, tuple[str, ...]] = {
    "credit_bureau_a_1": (
        "dpdmax_139P",
        "dpdmax_757P",
        "dpdmaxdateyear_596T",
        "overdueamountmax_155A",
        "overdueamountmax_35A",
        "numberofoverdueinstlmax_1151L",
        "numberofoverdueinstlmax_1039L",
        "numberofinstls_320L",
        "numberofinstls_229L",
        "residualamount_856A",
        "residualamount_488A",
        "totaloutstanddebtvalue_39A",
        "totaloutstanddebtvalue_668A",
        "debtoutstand_525A",
        "debtoverdue_47A",
    ),
    "credit_bureau_a_2": (
        "pmts_dpd_1073P",
        "pmts_dpd_303P",
        "pmts_overdue_1140A",
        "pmts_overdue_1152A",
        "pmts_month_158T",
        "pmts_month_706T",
        "pmts_year_1139T",
        "pmts_year_507T",
    ),
}


def split_dir(data_dir: Path, split: str) -> Path:
    return data_dir / "parquet_files" / split


def files_for_group(data_dir: Path, split: str, group: str) -> list[Path]:
    paths = sorted(split_dir(data_dir, split).glob(f"{split}_{group}*.parquet"))
    if not paths:
        raise FileNotFoundError(f"No parquet files found for split={split!r}, group={group!r}")
    return paths


def scan_group(data_dir: Path, split: str, group: str) -> pl.LazyFrame:
    paths = files_for_group(data_dir, split, group)
    frames = [pl.scan_parquet(str(path)) for path in paths]
    schemas = [frame.collect_schema() for frame in frames]
    target_schema: dict[str, pl.DataType] = {}

    for schema in schemas:
        for col, dtype in schema.items():
            if col not in target_schema or target_schema[col] == pl.Null:
                target_schema[col] = dtype
    for key in ("case_id", "num_group1", "num_group2"):
        if key in target_schema:
            target_schema[key] = pl.Int64
    for col, dtype in list(target_schema.items()):
        if dtype == pl.Null:
            target_schema[col] = pl.Utf8

    normalized = []
    for frame, schema in zip(frames, schemas):
        exprs = []
        for col, target_dtype in target_schema.items():
            if col in schema and schema[col] != target_dtype:
                exprs.append(pl.col(col).cast(target_dtype, strict=False).alias(col))
        normalized.append(frame.with_columns(exprs) if exprs else frame)
    return pl.concat(normalized, how="vertical_relaxed")


def normalize_dates(lf: pl.LazyFrame) -> pl.LazyFrame:
    schema = lf.collect_schema()
    exprs = []
    for name, dtype in schema.items():
        if name.endswith("D") or name == "date_decision":
            if dtype.is_numeric():
                continue
            exprs.append(
                pl.col(name)
                .cast(pl.Utf8)
                .str.strptime(pl.Date, strict=False)
                .cast(pl.Int32)
                .alias(name)
            )
    return lf.with_columns(exprs) if exprs else lf


def load_base(data_dir: Path, split: str) -> pl.LazyFrame:
    return normalize_dates(scan_group(data_dir, split, "base"))


def filter_cases(lf: pl.LazyFrame, case_ids: pl.LazyFrame | None) -> pl.LazyFrame:
    if case_ids is None:
        return lf
    return lf.join(case_ids, on="case_id", how="semi")


def load_depth0(
    data_dir: Path,
    split: str,
    group: str,
    case_ids: pl.LazyFrame | None = None,
) -> pl.LazyFrame:
    lf = normalize_dates(scan_group(data_dir, split, group))
    lf = filter_cases(lf, case_ids)
    return lf.unique(subset=["case_id"], keep="last")


def aggregate_group(
    data_dir: Path,
    split: str,
    group: str,
    case_ids: pl.LazyFrame | None = None,
) -> pl.LazyFrame:
    lf = normalize_dates(scan_group(data_dir, split, group))
    lf = filter_cases(lf, case_ids)
    schema = lf.collect_schema()
    aggs: list[pl.Expr] = [pl.len().alias(f"{group}__row_count")]

    for col, dtype in schema.items():
        if col in SPECIAL_COLUMNS:
            continue
        out = f"{group}__{col}"
        if dtype.is_numeric() or dtype == pl.Boolean or col.endswith("D"):
            base = pl.col(col).cast(pl.Float64, strict=False)
            aggs.extend(
                [
                    base.mean().alias(f"{out}__mean"),
                    base.max().alias(f"{out}__max"),
                    base.min().alias(f"{out}__min"),
                    base.std().alias(f"{out}__std"),
                ]
            )
        else:
            aggs.append(pl.col(col).n_unique().alias(f"{out}__nunique"))

    return lf.group_by("case_id").agg(aggs)


def aggregate_large_group_lite(
    data_dir: Path,
    split: str,
    group: str,
    case_ids: pl.LazyFrame | None = None,
) -> pl.LazyFrame:
    """Aggregate very large groups file-by-file using only cheap composable stats.

    This avoids the high memory peak of grouping all credit_bureau_a_2 rows at once.
    """
    selected = LITE_LARGE_NUMERIC_COLUMNS[group]
    partials: list[pl.LazyFrame] = []
    for path in files_for_group(data_dir, split, group):
        lf = pl.scan_parquet(str(path))
        schema = lf.collect_schema()
        available = [col for col in selected if col in schema]
        lf = lf.select(["case_id", *available]).with_columns(pl.col("case_id").cast(pl.Int64))
        lf = filter_cases(lf, case_ids)
        aggs: list[pl.Expr] = [pl.len().alias(f"{group}__row_count")]
        for col in available:
            base = pl.col(col).cast(pl.Float64, strict=False)
            prefix = f"{group}__{col}"
            aggs.extend(
                [
                    base.sum().alias(f"{prefix}__sum"),
                    base.count().alias(f"{prefix}__count"),
                    base.max().alias(f"{prefix}__max"),
                    base.min().alias(f"{prefix}__min"),
                ]
            )
        partials.append(lf.group_by("case_id").agg(aggs))

    partial = pl.concat(partials, how="vertical_relaxed")
    schema = partial.collect_schema()
    final_aggs: list[pl.Expr] = [pl.col(f"{group}__row_count").sum().alias(f"{group}__row_count")]
    for col in selected:
        prefix = f"{group}__{col}"
        sum_col = f"{prefix}__sum"
        count_col = f"{prefix}__count"
        max_col = f"{prefix}__max"
        min_col = f"{prefix}__min"
        if sum_col not in schema:
            continue
        final_aggs.extend(
            [
                (pl.col(sum_col).sum() / pl.col(count_col).sum()).alias(f"{prefix}__mean"),
                pl.col(max_col).max().alias(f"{prefix}__max"),
                pl.col(min_col).min().alias(f"{prefix}__min"),
            ]
        )
    return partial.group_by("case_id").agg(final_aggs)


def aggregate_large_group_lite_eager(
    data_dir: Path,
    split: str,
    group: str,
    temp_dir: Path,
    case_ids: pl.DataFrame | None = None,
) -> pl.DataFrame:
    selected = LITE_LARGE_NUMERIC_COLUMNS[group]
    temp_dir.mkdir(parents=True, exist_ok=True)
    partial_paths: list[Path] = []
    case_ids_lf = case_ids.lazy() if case_ids is not None else None

    for idx, path in enumerate(files_for_group(data_dir, split, group)):
        schema = pl.scan_parquet(str(path)).collect_schema()
        available = [col for col in selected if col in schema]
        lf = pl.scan_parquet(str(path)).select(["case_id", *available])
        lf = lf.with_columns(pl.col("case_id").cast(pl.Int64))
        lf = filter_cases(lf, case_ids_lf)
        aggs: list[pl.Expr] = [pl.len().alias(f"{group}__row_count")]
        for col in available:
            base = pl.col(col).cast(pl.Float64, strict=False)
            prefix = f"{group}__{col}"
            aggs.extend(
                [
                    base.sum().alias(f"{prefix}__sum"),
                    base.count().alias(f"{prefix}__count"),
                    base.max().alias(f"{prefix}__max"),
                    base.min().alias(f"{prefix}__min"),
                ]
            )
        partial = lf.group_by("case_id").agg(aggs).collect(engine="streaming")
        out_path = temp_dir / f"{split}_{group}_partial_{idx}.parquet"
        partial.write_parquet(out_path)
        partial_paths.append(out_path)
        del partial

    partial_lf = pl.scan_parquet([str(path) for path in partial_paths])
    schema = partial_lf.collect_schema()
    final_aggs: list[pl.Expr] = [pl.col(f"{group}__row_count").sum().alias(f"{group}__row_count")]
    for col in selected:
        prefix = f"{group}__{col}"
        sum_col = f"{prefix}__sum"
        count_col = f"{prefix}__count"
        max_col = f"{prefix}__max"
        min_col = f"{prefix}__min"
        if sum_col not in schema:
            continue
        final_aggs.extend(
            [
                (pl.col(sum_col).sum() / pl.col(count_col).sum()).alias(f"{prefix}__mean"),
                pl.col(max_col).max().alias(f"{prefix}__max"),
                pl.col(min_col).min().alias(f"{prefix}__min"),
            ]
        )
    return partial_lf.group_by("case_id").agg(final_aggs).collect(engine="streaming")


def build_features(
    data_dir: Path,
    split: str,
    preset_name: str,
    cache_dir: Path | None = None,
    use_cache: bool = True,
    sample_rows: int = 0,
) -> pl.DataFrame:
    if preset_name not in PRESETS:
        raise ValueError(f"Unknown preset {preset_name!r}. Available: {sorted(PRESETS)}")
    preset = PRESETS[preset_name]

    cache_path = None
    if cache_dir is not None and not sample_rows:
        cache_path = cache_dir / f"{split}_{preset_name}_features.parquet"
        if use_cache and cache_path.exists():
            return pl.read_parquet(cache_path)

    lf = load_base(data_dir, split)
    if sample_rows:
        lf = lf.sort("case_id").head(sample_rows)
    case_ids = lf.select("case_id")
    for group in preset.depth0_groups:
        lf = lf.join(load_depth0(data_dir, split, group, case_ids), on="case_id", how="left")
    for group in preset.aggregate_groups:
        lf = lf.join(aggregate_group(data_dir, split, group, case_ids), on="case_id", how="left")
    df = lf.collect(engine="streaming")
    if preset.lite_large_groups:
        case_ids_df = df.select("case_id")
        temp_root = (cache_path.parent if cache_path is not None else Path("outputs/features")) / "_tmp_lite"
        for group in preset.lite_large_groups:
            lite = aggregate_large_group_lite_eager(
                data_dir,
                split,
                group,
                temp_root / f"{split}_{group}",
                case_ids=case_ids_df if sample_rows else None,
            )
            df = df.join(lite, on="case_id", how="left")
    if cache_path is not None and not sample_rows:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        df.write_parquet(cache_path)
    return df

In [ ]:
DROP_COLUMNS = {"case_id", "target"}


def to_pandas(df: pl.DataFrame) -> pd.DataFrame:
    pdf = df.to_pandas()
    for col in pdf.columns:
        if pd.api.types.is_bool_dtype(pdf[col]):
            pdf[col] = pdf[col].astype("int8")
        elif is_float_dtype(pdf[col]):
            pdf[col] = pdf[col].astype("float32")
        elif is_integer_dtype(pdf[col]) and col != "case_id":
            pdf[col] = pd.to_numeric(pdf[col], downcast="integer")
    return pdf


def fit_category_maps(frame: pd.DataFrame, columns: list[str]) -> dict[str, list[object]]:
    maps = {}
    for col in columns:
        if pd.api.types.is_object_dtype(frame[col]) or pd.api.types.is_string_dtype(frame[col]):
            cat = frame[col].astype("category")
            maps[col] = list(cat.cat.categories)
            frame[col] = cat.cat.codes.astype("int32")
    return maps


def apply_category_maps(frame: pd.DataFrame, maps: dict[str, list[object]]) -> None:
    for col, categories in maps.items():
        if col in frame.columns:
            mapper = {value: idx for idx, value in enumerate(categories)}
            frame[col] = frame[col].map(mapper).fillna(-1).astype("int32")
    for col in frame.columns:
        if pd.api.types.is_object_dtype(frame[col]) or pd.api.types.is_string_dtype(frame[col]):
            frame[col] = frame[col].astype("category").cat.codes.astype("int32")


def align_columns(feature_cols: list[str], frame: pd.DataFrame) -> pd.DataFrame:
    missing = [col for col in feature_cols if col not in frame.columns]
    if missing:
        frame = pd.concat([frame, pd.DataFrame(np.nan, index=frame.index, columns=missing)], axis=1)
    extra = [col for col in frame.columns if col not in feature_cols]
    if extra:
        frame = frame.drop(columns=extra)
    return frame[feature_cols]


def lgbm_params(n_estimators: int, seed: int = 42) -> dict:
    return {
        "objective": "binary",
        "n_estimators": int(n_estimators),
        "learning_rate": 0.03,
        "num_leaves": 96,
        "max_depth": -1,
        "min_child_samples": 80,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.75,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "random_state": seed,
        "n_jobs": -1,
        "device_type": "cpu",
        "verbosity": -1,
    }

In [ ]:
def train_predict_stage(preset: str, n_estimators: int, seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    print(f"=== stage {preset} n_estimators={n_estimators} ===")
    check_memory(f"{preset}: start", MAX_RSS_GB, MIN_AVAILABLE_GB)

    train_pl = build_features(
        ROOT,
        "train",
        preset,
        cache_dir=WORKING / "features",
        use_cache=True,
        sample_rows=SAMPLE_ROWS,
    )
    print(f"{preset}: train shape", train_pl.shape)
    check_memory(f"{preset}: after build train features", MAX_RSS_GB, MIN_AVAILABLE_GB)

    train_pdf = to_pandas(train_pl)
    del train_pl
    gc.collect()
    check_memory(f"{preset}: after train to pandas", MAX_RSS_GB, MIN_AVAILABLE_GB)

    y = train_pdf["target"].astype("int8")
    feature_cols = [col for col in train_pdf.columns if col not in DROP_COLUMNS]
    category_maps = fit_category_maps(train_pdf, feature_cols)
    for col in feature_cols:
        if is_float_dtype(train_pdf[col]):
            train_pdf[col] = train_pdf[col].astype("float32")
        elif is_integer_dtype(train_pdf[col]):
            train_pdf[col] = pd.to_numeric(train_pdf[col], downcast="integer")
    X = train_pdf[feature_cols]
    check_memory(f"{preset}: before fit", MAX_RSS_GB, MIN_AVAILABLE_GB)

    model = lgb.LGBMClassifier(**lgbm_params(n_estimators=n_estimators, seed=seed))
    model.fit(X, y)

    del X, y, train_pdf
    gc.collect()
    check_memory(f"{preset}: after release train", MAX_RSS_GB, MIN_AVAILABLE_GB)

    test_pl = build_features(
        ROOT,
        "test",
        preset,
        cache_dir=WORKING / "features",
        use_cache=True,
    )
    print(f"{preset}: test shape", test_pl.shape)
    test_pdf = to_pandas(test_pl)
    del test_pl
    gc.collect()

    test_ids = test_pdf["case_id"].to_numpy()
    test_features = test_pdf.drop(columns=[col for col in DROP_COLUMNS if col in test_pdf.columns])
    X_test = align_columns(feature_cols, test_features)
    apply_category_maps(X_test, category_maps)
    del test_pdf, test_features
    gc.collect()
    check_memory(f"{preset}: before predict", MAX_RSS_GB, MIN_AVAILABLE_GB)

    pred = model.predict_proba(X_test)[:, 1].astype(np.float32)
    del model, X_test
    gc.collect()
    check_memory(f"{preset}: done", MAX_RSS_GB, MIN_AVAILABLE_GB)
    return test_ids, pred

In [ ]:
if DRY_RUN:
    stages = [("static", 71, 42, 1.0)]
else:
    stages = [
        ("medium", 537, 42, 0.5),
        ("a2ultra", 868, 42, 0.5),
    ]

final_ids = None
final_pred = None
for preset, n_estimators, seed, weight in stages:
    ids, pred = train_predict_stage(preset, n_estimators, seed)
    if final_ids is None:
        final_ids = ids
        final_pred = np.zeros(len(pred), dtype=np.float32)
    final_pred += weight * pred

submission = pd.DataFrame({"case_id": final_ids, "score": final_pred})
submission = sample_submission[["case_id"]].merge(submission, on="case_id", how="left")
submission["score"] = submission["score"].fillna(float(np.mean(final_pred)))
submission.to_csv(WORKING / "submission.csv", index=False)
submission.to_csv("submission.csv", index=False)
check_memory("submission done", MAX_RSS_GB, MIN_AVAILABLE_GB)
print(submission.shape)
submission.head()